#Task 1

In [8]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

import kagglehub
from kagglehub import KaggleDatasetAdapter

# Set the path to the file you'd like to load
file_path = "/Users/Admin/Desktop"

# Load the latest version
df = kagglehub.load_dataset(
  KaggleDatasetAdapter.PANDAS,
  "mervemenekse/ecommerce-dataset",
  file_path,

_IncompleteInputError: incomplete input (1111859083.py, line 15)

In [ ]:
# Handle the null values
df.dropna(subset=['CustomerID'], inplace=True)
# Fix data types
df['InvoiceDate'] = pd.to_dateline(df['InvoiceDate'])
# Remove ivalid data
df = df[(df['Quantity'] > 0) & (df['UnitPrice'] > 0)]
# Remove duplicates
df.drop_duplicates(inplace=True)

#Task 2

In [ ]:
# Create Invoice Month
df['InvoiceMonth'] = df['InvoiceDate'].dt.to_period('M')

# Group by CustomerID to find their first purchase month
df['CohortMonth'] = df.groupby('CustomerID')['InvoiceMonth'].transform('min')

# Calculate the month difference (CohortIndex)
def get_date_int(df, column):
    year = df[column].dt.year
    month = df[column].dt.month
    return year, month

inv_year, inv_month = get_date_int(df, 'InvoiceMonth')
cohort_year, cohort_month = get_date_int(df, 'CohortMonth')

years_diff = inv_year - cohort_year
months_diff = inv_month - cohort_month
df['CohortIndex'] = years_diff * 12 + months_diff + 1

# Pivot for the Heatmap
cohort_counts = df.groupby(['CohortMonth', 'CohortIndex'])['CustomerID'].nunique().reset_index()
cohort_pivot = cohort_counts.pivot(index='CohortMonth', columns='CohortIndex', values='CustomerID')

# Calculate retention percentage
cohort_sizes = cohort_pivot.iloc[:, 0]
retention = cohort_pivot.divide(cohort_sizes, axis=0)

plt.figure(figsize=(12, 8))
sns.heatmap(retention, annot=True, fmt='.0%', cmap='YlGnBu')
plt.title('Customer Retention by Cohort')
plt.show()


# Task 3

In [ ]:
# Feature Engineering
df['TotalSales'] = df['Quantity'] * df['UnitPrice']
df['Hour'] = df['InvoiceDate'].dt.hour

# Selecting numerical features for correlation
corr_data = df[['TotalSales', 'Quantity', 'UnitPrice', 'Hour']]
correlation_matrix = corr_data.corr()

plt.figure(figsize=(8, 6))
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', center=0)
plt.title('Feature Correlation Heatmap')
plt.show()